# Set up and Get data

In [39]:
!pip install google-cloud-bigquery openpyxl db-dtypes google-cloud-firestore -qq 

In [40]:
import calendar
from datetime import date
import os
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from google.cloud import firestore
from google.cloud import bigquery

In [41]:
def generate_month_query(table_full_name: str, tz_name: str = "Asia/Ho_Chi_Minh", ref_date: datetime | None = None) -> str:
    """
    Trả về query cho "tháng hiện tại" theo timezone tz_name.
    table_full_name ví dụ: "real-estate-project-445516.real_estate_data.ads_data"
    ref_date dùng để test (nếu None thì lấy thời điểm hiện tại).
    """
    tz = ZoneInfo(tz_name)
    now = ref_date.astimezone(tz) if ref_date else datetime.now(tz)
    year = now.year
    month = now.month

    # ngày bắt đầu tháng (00:00:00) và ngày cuối tháng (00:00:00)
    first_day_str = f"{year:04d}-{month:02d}-01T00:00:00"
    last_day_num = calendar.monthrange(year, month)[1]
    last_day_str = f"{year:04d}-{month:02d}-{last_day_num:02d}T00:00:00"

    query = f"""select *
                from `{table_full_name}`
                where (`Ngày đăng` <= '{first_day_str}' and `Ngày hết hạn` >= '{first_day_str}') or
                      (`Ngày đăng` >= '{first_day_str}' and `Ngày đăng` <= '{last_day_str}');"""
    return query


In [42]:
client = bigquery.Client.from_service_account_json("/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json")
table = "real-estate-project-445516.real_estate_data.ads_data"
query = generate_month_query(table)

In [43]:
df = client.query(query).to_dataframe()

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [44]:
# df = pd.read_csv("/lakehouse/default/Files/ads_data_2025_11(1).csv")

In [45]:
def safe_mean_round(s):
    values = [x for x in s if pd.notna(x)]
    return round(float(np.nanmean(values)), 2) if values else None

In [46]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/lakehouse/default/Files/real-estate-project-445516-firebase-adminsdk-fbsvc-8d09bd7be7.json" 
db = firestore.Client()  # sẽ dùng GOOGLE_APPLICATION_CREDENTIALS env var

In [47]:
# def count_documents():
#     coll_ref = db.collection("price_data")
#     aggregation_query = coll_ref.count()

#     result = aggregation_query.get()
#     print("Total documents:", result[0])

# count_documents()

In [48]:
# bỏ những dòng không có giá
df1 = df.dropna(subset=['Khoảng giá']).copy()

In [49]:
group_cols = [ 
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án', 
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào' 
]

agg_df = (
    df1
    .groupby(group_cols, dropna=False)
    .agg(
        distinct_khoang_gia = ('Khoảng giá', lambda s: np.sort(s.unique()).tolist()),
        mean_unique_khoang_gia = ('Khoảng giá', lambda s: np.mean(np.unique(s)) if len(np.unique(s))>0 else np.nan),
        unique_ads = ('Mã tin', lambda s: s.unique().tolist()),
        n_unique_prices = ('Khoảng giá', lambda s: len(np.unique(s)))
    )
    .reset_index()
)

# nếu muốn chuyển mean sang triệu hoặc tỷ (ví dụ sang triệu VND):
agg_df['mean_unique_khoang_gia_million'] = agg_df['mean_unique_khoang_gia'] / 1e6
# hoặc sang tỷ:
agg_df['mean_unique_khoang_gia_billion'] = agg_df['mean_unique_khoang_gia'] / 1e9
# Tính giá mỗi m2
agg_df = agg_df[agg_df["Diện tích"] != 0]
agg_df["mean_unique_khoang_gia_million/m2"] = agg_df.apply(
    lambda row: row["mean_unique_khoang_gia_million"] / row["Diện tích"] if row["Diện tích"] not in [0, None] else None,
    axis=1
)
agg_df["mean_unique_khoang_gia_billion/m2"] = agg_df.apply(
    lambda row: row["mean_unique_khoang_gia_billion"] / row["Diện tích"] if row["Diện tích"] not in [0, None] else None,
    axis=1
)


In [50]:
agg_df.shape

(242561, 22)

In [51]:
# làm tròn chữ cái đầu tiên của tất cả các giá trị trong cột bds_type
agg_df['Loại BĐS'] = agg_df['Loại BĐS'].str.strip().str.capitalize()

In [52]:
ban_df = agg_df[agg_df["Loại quảng cáo"] == "Bán"]
thue_df = agg_df[agg_df["Loại quảng cáo"] == "Cho thuê"]

In [53]:
# Lấy tháng hiện tại
year_month = datetime.now().strftime("%Y-%m")

# Lấy tháng trước
today = date.today()
if today.month == 1:
    prev_year = today.year - 1
    prev_month = 12
else:
    prev_year = today.year
    prev_month = today.month - 1
prev_year_month = f"{prev_year}-{prev_month:02d}"

In [54]:
# year_month = "2025-11"
# prev_year_month = "2025-10"

# Loại bỏ nhiễu

In [55]:
# bỏ những dòng không thực sự thuộc nhà riêng
agg_df["so_tang"] = agg_df["Số tầng"].str.extract(r'(\d+)')
agg_df["so_tang"] = pd.to_numeric(agg_df["so_tang"], errors='coerce')
agg_df = agg_df[(agg_df["Diện tích"]<=150) | (agg_df["Loại BĐS"] != "nhà riêng") | (agg_df["so_tang"] <= 3)]

In [56]:
price_col = 'mean_unique_khoang_gia_million/m2'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in ban_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
ban_df = ban_df.drop(index=idx_to_drop).reset_index(drop=True)

In [57]:
price_col = 'mean_unique_khoang_gia_million'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in thue_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
thue_df = thue_df.drop(index=idx_to_drop).reset_index(drop=True)

In [58]:
ban_df.shape

(196876, 22)

In [59]:
thue_df.shape

(40670, 22)

# Tính giá BĐS Bán

## Group by tới Quận 

In [60]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận']

grouped = (
    ban_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million_per_m2 = ('mean_unique_khoang_gia_million/m2', safe_mean_round),
        avg_price_billion_per_m2 = ('mean_unique_khoang_gia_billion/m2', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
    'Quận': 'district'
}
grouped = grouped.rename(columns=rename_map)


In [61]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million_per_m2') is None or (isinstance(row.get('avg_price_million_per_m2'), float) and np.isnan(row.get('avg_price_million_per_m2'))) else float(row.get('avg_price_million_per_m2')),
        "avg_price_billion": None if row.get('avg_price_billion_per_m2') is None or (isinstance(row.get('avg_price_billion_per_m2'), float) and np.isnan(row.get('avg_price_billion_per_m2'))) else float(row.get('avg_price_billion_per_m2')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 1610 documents to Firestore collection 'price_data' in 5 batch(es).


## Group by tới Tỉnh thành 

In [62]:
# --- 2) group lại theo 3 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố']

grouped = (
    ban_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million_per_m2 = ('mean_unique_khoang_gia_million/m2', safe_mean_round),
        avg_price_billion_per_m2 = ('mean_unique_khoang_gia_billion/m2', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
}
grouped = grouped.rename(columns=rename_map)


In [63]:
# thêm cột Quận 
grouped['district'] = "All"

In [64]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million_per_m2') is None or (isinstance(row.get('avg_price_million_per_m2'), float) and np.isnan(row.get('avg_price_million_per_m2'))) else float(row.get('avg_price_million_per_m2')),
        "avg_price_billion": None if row.get('avg_price_billion_per_m2') is None or (isinstance(row.get('avg_price_billion_per_m2'), float) and np.isnan(row.get('avg_price_billion_per_m2'))) else float(row.get('avg_price_billion_per_m2')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 379 documents to Firestore collection 'price_data' in 1 batch(es).


# Tính giá BĐS cho thuê

## Group by tới Quận 

In [65]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận']

grouped = (
    thue_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million = ('mean_unique_khoang_gia_million', safe_mean_round),
        avg_price_billion = ('mean_unique_khoang_gia_billion', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
    'Quận': 'district'
}
grouped = grouped.rename(columns=rename_map)


In [66]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million') is None or (isinstance(row.get('avg_price_million'), float) and np.isnan(row.get('avg_price_million'))) else float(row.get('avg_price_million')),
        "avg_price_billion": None if row.get('avg_price_billion') is None or (isinstance(row.get('avg_price_billion'), float) and np.isnan(row.get('avg_price_billion'))) else float(row.get('avg_price_billion')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 712 documents to Firestore collection 'price_data' in 2 batch(es).


## Group by tới Tỉnh thành 

In [67]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố']

grouped = (
    thue_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million = ('mean_unique_khoang_gia_million', safe_mean_round),
        avg_price_billion = ('mean_unique_khoang_gia_billion', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
}
grouped = grouped.rename(columns=rename_map)


In [68]:
# thêm cột Quận 
grouped['district'] = "All"

In [69]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million') is None or (isinstance(row.get('avg_price_million'), float) and np.isnan(row.get('avg_price_million'))) else float(row.get('avg_price_million')),
        "avg_price_billion": None if row.get('avg_price_billion') is None or (isinstance(row.get('avg_price_billion'), float) and np.isnan(row.get('avg_price_billion'))) else float(row.get('avg_price_billion')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")

Pushed 181 documents to Firestore collection 'price_data' in 1 batch(es).


# Tính metric trên trang chủ

In [79]:
def push_to_firestore(grouped: pd.DataFrame, metric_name: str, collection_name = "metrics"):
    # Lấy giá tháng trước
    docs = (
        db.collection("metrics")
        .where("year_month", "==", prev_year_month)
        .stream()
    )
    LM_values = [doc.to_dict() for doc in docs]
    LM_df = pd.DataFrame(LM_values)

    # thêm cột cho grouped
    grouped['year_month'] = year_month
    grouped['name'] = metric_name

    # rename columns
    rename_map = {
        'Tỉnh thành phố': 'province',
    }
    grouped = grouped.rename(columns=rename_map)

    # --- thêm dòng All ---
    total_value = grouped['value'].sum()
    all_row = pd.DataFrame([{
        "name": metric_name,
        "province": "All",
        "year_month": year_month,
        "value": total_value
    }])
    grouped = pd.concat([grouped, all_row], ignore_index=True)
    
    # Tính % so với LM
    if LM_df.empty:
        grouped['vs_LM'] = 0
    else:
        LM_merge = LM_df[['name', 'province', 'value']].rename(columns={'value': 'value_LM'})
        grouped = grouped.merge(LM_merge, on=["name", 'province'], how='left')
        grouped['vs_LM'] = np.where(
            grouped['value_LM'].isna(),
            0,
            np.where(
                grouped['value_LM'] == 0,
                0,
                (grouped['value'] - grouped['value_LM']) / grouped['value_LM'] * 100
            )
        )
        grouped['vs_LM'] = grouped['vs_LM'].round(2)

    # Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
    batch = db.batch()
    batch_size = 400
    written = 0
    committed_batches = 0

    for idx, row in grouped.iterrows():
        # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
        raw_id = f"{row.get('name','')}/{row.get('province','')}/{row['year_month']}"
        doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

        doc_ref = db.collection(collection_name).document(doc_id)

        # payload: chuyển các giá trị numpy -> native python
        payload = {
            "name": None if pd.isna(row.get('name')) else str(row.get('name')),
            "province": None if pd.isna(row.get('province')) else str(row.get('province')),
            "year_month": row['year_month'],
            "value": None if pd.isna(row.get('value')) else float(row.get('value')) if row.get('name') == "total_sales_amt" else int(row.get('value')),
            "vs_LM": None if pd.isna(row.get('vs_LM')) else str(row.get('vs_LM')),
            "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
        }

        # set (merge True để upsert - update nếu tồn tại)
        batch.set(doc_ref, payload, merge=True)
        written += 1

        if written % batch_size == 0:
            batch.commit()
            committed_batches += 1
            batch = db.batch()

    # commit remaining
    if written % batch_size != 0:
        batch.commit()
        committed_batches += 1

    print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")

## Tính số dự án


In [80]:
metric_name = "project_count"
grouped = (
    agg_df
        .groupby("Tỉnh thành phố")["Tên dự án"]
        .nunique()
        .reset_index(name="value")
)
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:315: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Pushed 64 documents to Firestore collection 'metrics' in 1 batch(es).


## Tính số bất động sản bán

In [81]:
metric_name = "sales_bds_count"
grouped = agg_df[agg_df["Loại quảng cáo"] == "Bán"].groupby("Tỉnh thành phố").size().reset_index(name="value")
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:315: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Pushed 64 documents to Firestore collection 'metrics' in 1 batch(es).


## Tính tổng giá trị

In [82]:
metric_name = "total_sales_amt"
grouped = agg_df[agg_df["Loại quảng cáo"] == "Bán"].groupby("Tỉnh thành phố")["mean_unique_khoang_gia_billion"].sum().reset_index(name="value")
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:315: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Pushed 64 documents to Firestore collection 'metrics' in 1 batch(es).


# Xóa doc

In [83]:
# !pip install google-cloud-firestore -qq 
# import time
# from google.cloud import firestore
# from google.api_core import exceptions

# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/lakehouse/default/Files/real-estate-project-445516-firebase-adminsdk-fbsvc-8d09bd7be7.json" 
# db = firestore.Client()  # sẽ dùng GOOGLE_APPLICATION_CREDENTIALS env var

# def delete_collection(collection_path: str, batch_size: int = 500, max_retries: int = 5, sleep_between_batches: float = 0.1):
#     coll_ref = db.collection(collection_path)
#     # coll_ref = db.collection(collection_path).where("year_month", "==", "2025-12")

#     while True:
#         # Lấy 1 page
#         attempt = 0
#         docs = None
#         while attempt <= max_retries:
#             try:
#                 docs = coll_ref.limit(batch_size).get()   # tránh .stream()
#                 break
#             except (exceptions.DeadlineExceeded, exceptions.ServiceUnavailable, exceptions.InternalServerError) as e:
#                 attempt += 1
#                 wait = 2 ** attempt
#                 print(f"Get attempt {attempt} failed: {e}. retrying in {wait}s...")
#                 time.sleep(wait)
#         if docs is None:
#             raise RuntimeError("Failed to fetch documents after retries.")

#         if not docs:
#             print("No more documents — finished.")
#             return

#         # Xóa theo batch
#         batch = db.batch()
#         for d in docs:
#             batch.delete(d.reference)

#         attempt = 0
#         while attempt <= max_retries:
#             try:
#                 batch.commit()
#                 print(f"Deleted batch of {len(docs)} documents.")
#                 break
#             except (exceptions.DeadlineExceeded, exceptions.ServiceUnavailable, exceptions.InternalServerError) as e:
#                 attempt += 1
#                 wait = 2 ** attempt
#                 print(f"Commit attempt {attempt} failed: {e}. retrying in {wait}s...")
#                 time.sleep(wait)
#         else:
#             raise RuntimeError("Failed to commit delete batch after retries.")

#         time.sleep(sleep_between_batches)

# delete_collection("price_data")